In [1]:
import json
import ctypes
import dask.dataframe as dd
from dask.distributed import Client
import pandas as pd

In [2]:
def trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)


def extract_helpful_votes(helpful_str_col, idx):
    return helpful_str_col.map_partitions(
        lambda part: part.apply(
            lambda x: str(x).strip("[]").split(", ")[idx] if pd.notna(x) and len(str(x).strip("[]").split(", ")) > idx else "0"
        ),
        meta=(f"helpful_{idx}", "object"),
    ).astype(int)


def extract_year(review_time_col):
    return review_time_col.map_partitions(
        lambda part: part.apply(
            lambda x: x.split(", ")[-1] if pd.notna(x) and ", " in x else None
        ),
        meta=("reviewing_since", "object"),
    ).astype("Int64")  # handle missing years safely

In [3]:
client = Client()
client.run(trim_memory)

{'tcp://127.0.0.1:33973': 1,
 'tcp://127.0.0.1:35333': 1,
 'tcp://127.0.0.1:38729': 1,
 'tcp://127.0.0.1:39135': 1}

In [6]:
user_reviews_dd = dd.read_csv("user_reviews.csv")
user_reviews_dd = user_reviews_dd.sample(frac=0.01).persist()

In [7]:
len(user_reviews_dd)

496016